# 02 — Preprocessing and dataset split

Creates a reproducible stratified 70/15/15 train, validation, and test manifest. The original image files stay untouched.

In [3]:

from pathlib import Path
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Step 1: Set random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Step 2: Define correct paths
ROOT = Path(
    "/Users/thakkifilyas/Desktop/DL_Project/plant-disease-repo"
)

DATASET_DIR = Path(
    "/Users/thakkifilyas/Desktop/DL_Project/PlantVillage"
)

# Step 3: Supported image extensions
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".gif", ".tif", ".tiff"
}

print("Project root:", ROOT)
print("Dataset:", DATASET_DIR)

# Step 4: Verify dataset directory
assert DATASET_DIR.is_dir(), (
    f"Dataset folder not found: {DATASET_DIR}"
)

# Step 5: Collect image paths and class labels
records = []

for class_dir in sorted(DATASET_DIR.iterdir()):
    if class_dir.is_dir():
        for path in sorted(class_dir.rglob("*")):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({
                    "path": str(path.resolve()),
                    "class": class_dir.name
                })

# Step 6: Create DataFrame
df = pd.DataFrame(records)

assert len(df) > 0, "No images found"

print("Total images:", len(df))
print("Total classes:", df["class"].nunique())

# Step 7: Split into 70% training and 30% temporary
train, temp = train_test_split(
    df,
    test_size=0.30,
    stratify=df["class"],
    random_state=SEED
)

# Step 8: Split temporary data equally
# 15% validation + 15% testing
val, test = train_test_split(
    temp,
    test_size=0.50,
    stratify=temp["class"],
    random_state=SEED
)

# Step 9: Label each split
train = train.assign(split="train")
val = val.assign(split="validation")
test = test.assign(split="test")

# Step 10: Combine datasets into one manifest
manifest = pd.concat(
    [train, val, test],
    ignore_index=True
)

manifest = manifest.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

# Step 11: Save preprocessing results
out = ROOT / "results" / "preprocessing"
out.mkdir(parents=True, exist_ok=True)

manifest.to_csv(
    out / "split_manifest.csv",
    index=False
)

# Step 12: Display class distribution
print("\nClass distribution:")
print(
    manifest.groupby(["split", "class"])
    .size()
    .unstack(fill_value=0)
)

# Step 13: Display split percentages
print("\nDataset split percentages:")
print(
    manifest["split"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nDataset splitting completed successfully!")
print("Manifest saved to:", out / "split_manifest.csv")


Project root: /Users/thakkifilyas/Desktop/DL_Project/plant-disease-repo
Dataset: /Users/thakkifilyas/Desktop/DL_Project/PlantVillage
Total images: 4172
Total classes: 15

Class distribution:
class       Pepper__bell___Bacterial_spot  Pepper__bell___healthy  \
split                                                               
test                                   40                       8   
train                                 185                      38   
validation                             39                       8   

class       Potato___Early_blight  Potato___Late_blight  Potato___healthy  \
split                                                                       
test                           43                    50                22   
train                         197                   234               107   
validation                     42                    50                23   

class       Tomato_Bacterial_spot  Tomato_Early_blight  Tomato_Late_blight  \

The manifest is the single source of truth for all later notebooks. The test split is kept separate and is used only for final evaluation.